<a href="https://colab.research.google.com/github/StillWork/BioAI/blob/main/19_LLM%E1%84%8B%E1%85%A6%E1%84%8B%E1%85%B5%E1%84%8C%E1%85%A5%E1%86%AB%E1%84%90%E1%85%B3_RAG_%E1%84%89%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%A3%E1%86%A8FDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 19 · LLM 에이전트 & RAG — 신약개발·FDA 규제 대응 도구
### 대화형 LLM 을 '똑똑한 연구 조수' 로 쓰는 법


지금 제약바이오 현장에서 가장 빠르게 도입되는 AI 는 사실 **대화형 LLM**
(ChatGPT·Gemini·Claude)입니다. 논문 요약, 규제 문서 해석, 실험 설계 아이디어,
코드 작성까지 — 연구자의 **범용 조수**로 쓰입니다.
하지만 LLM 은 최신·사내 문서를 모르고 때때로 **그럴듯한 거짓(할루시네이션)**
을 만듭니다. 이를 해결하는 표준 기술이 **RAG**(검색증강생성)입니다:
믿을 수 있는 문서에서 **먼저 근거를 찾아** LLM 에게 함께 주어, 근거에 기반한
답을 만들게 합니다.

**이 노트북에서 배우는 것**
1. LLM(Gemini) 을 코드로 호출해 **신약개발 Q&A** 하기
2. 우리 문서(FDA 가이던스·GMP·논문 요지)로 **미니 지식베이스** 만들기
3. 질문과 관련된 근거를 **임베딩 검색(RAG 의 R)** 으로 찾기
4. 찾은 근거를 넣어 **출처 기반 답변(RAG 의 G)** 생성 — 규제 대응 시나리오
5. LLM 을 **여러 도구를 쓰는 에이전트**로 확장하는 개념

> 💡 검색(2~3단계)은 인터넷/키 없이 **CPU 로 바로** 동작합니다.
> LLM 답변(1·4단계)은 무료 **Gemini API 키**가 필요합니다(발급 방법 아래).


## 0. 준비 — 라이브러리 설치
- `sentence-transformers`: 문장을 벡터로 바꿔 **의미 검색**에 사용(오프라인)
- `google-generativeai`: 무료 Gemini API 호출용


In [1]:
!pip install -q sentence-transformers google-generativeai
import numpy as np, textwrap
from sentence_transformers import SentenceTransformer
print("설치 완료 ✅")


설치 완료 ✅


In [18]:
!pip install -q rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 36.5 MB/s eta 0:00:00


## 1. ⭐ 대화형 LLM(Gemini) 호출하기
Google AI Studio(aistudio.google.com/app/apikey)에서 **무료 API 키**를 발급받아
아래에 붙여넣습니다. 강의에서 쓰던 Gemini 를 **코드로 자동화**하는 것입니다.

> 키가 없다면 이 셀은 건너뛰고, 뒤의 **검색(RAG-R)** 부분만 먼저 실행해도 됩니다.
> 그리고 만들어진 프롬프트를 복사해 Gemini/Claude 채팅창에 붙여넣어도 됩니다.


In [12]:
import getpass, google.generativeai as genai

API_KEY = getpass.getpass("Gemini API 키를 붙여넣으세요 (없으면 Enter): ").strip()
LLM = None

if API_KEY:
    genai.configure(api_key=API_KEY)
    try:
        # 사용 가능한 모델 목록에서 적합한 모델 찾기
        available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

        # 가장 범용적인 'gemini-1.5-flash' 시도
        target = "gemini-1.5-flash"
        # 모델 이름에 'models/'가 포함되어 있는지 확인하여 매칭
        matched = [m for m in available_models if target in m]

        if matched:
            model_to_use = matched[0]
        else:
            # 없을 경우 'flash'가 포함된 다른 모델 중 2.5/2.0 제외하고 선택
            flash_models = [m for m in available_models if "flash" in m.lower() and "2." not in m]
            model_to_use = flash_models[0] if flash_models else available_models[0]

        LLM = genai.GenerativeModel(model_to_use)
        print(f"사용 중인 모델: {model_to_use}")
    except Exception as e:
        print(f"모델 설정 중 오류 발생: {e}")

def ask_llm(prompt):
    if LLM is None:
        print("[키 없음] 아래 프롬프트를 Gemini/Claude 채팅에 붙여넣으세요:\n")
        print(prompt); return None
    try:
        return LLM.generate_content(prompt).text
    except Exception as e:
        return f"에러 발생: {e}"

# 간단한 신약개발 질문 테스트
ans = ask_llm("BACE-1 을 표적으로 하는 알츠하이머 신약개발의 핵심 과제를 3가지로 요약해줘.")
print(ans if ans else "")

Gemini API 키를 붙여넣으세요 (없으면 Enter): ··········


KeyboardInterrupt: 

## 2. 미니 지식베이스 만들기 (우리가 신뢰하는 문서)
RAG 의 핵심은 **믿을 수 있는 출처**입니다. 실제로는 사내 SOP, FDA 가이던스,
논문 PDF 등을 넣습니다. 여기서는 데모용으로 **규제·품질 관련 핵심 지식**을
짧은 문단(chunk)들로 준비합니다. (실무에선 PDF 를 잘라 이 리스트를 채웁니다)


In [13]:
KB = [
 {"id":"ALCOA+", "text":"데이터 무결성의 ALCOA+ 원칙: 데이터는 Attributable(귀속가능), "
   "Legible(판독가능), Contemporaneous(동시기록), Original(원본), Accurate(정확) 해야 하며, "
   "확장 요건으로 Complete(완전), Consistent(일관), Enduring(항구), Available(가용) 을 포함한다. "
   "FDA 483/경고서한에서 가장 자주 지적되는 영역이다."},
 {"id":"CSV/CSA", "text":"컴퓨터 시스템 밸리데이션(CSV)은 시스템이 의도대로 동작함을 문서로 입증한다. "
   "최근 FDA 는 위험기반의 CSA(Computer Software Assurance) 접근을 권장하여, 문서 부담을 줄이고 "
   "중요기능 테스트에 집중하도록 한다."},
 {"id":"공정밸리데이션","text":"공정 밸리데이션은 3단계로 구성된다: (1)공정설계, (2)공정성능적격성평가(PPQ), "
   "(3)지속적 공정검증(CPV). CPV 단계에서 SPC 관리도와 데이터 분석으로 상태를 상시 모니터링한다."},
 {"id":"세척밸리데이션","text":"세척 밸리데이션은 교차오염을 막기 위해 잔류물 허용한도(예: 독성기반 PDE/ADE)를 정하고 "
   "회수율을 검증한다. 여러 제품이 공용 설비를 쓸 때 필수다."},
 {"id":"안정성시험","text":"안정성 시험(ICH Q1A)은 온도·습도 조건에서 유효기간을 정한다. 가속(40C/75%RH)과 "
   "장기(25C/60%RH) 조건 데이터를 회귀분석하여 규격 이탈 시점을 예측한다."},
 {"id":"생동성","text":"생물학적 동등성(생동성)은 제네릭이 오리지널과 동등함을 입증한다. AUC 와 Cmax 의 "
   "기하평균비 90% 신뢰구간이 80~125% 이내여야 한다."},
 {"id":"OOS","text":"규격이탈(OOS) 결과는 조사(Phase I 실험실 오류 검토 → Phase II 생산 조사)를 거친다. "
   "머신러닝으로 배치 데이터를 학습하면 OOS 를 조기경보할 수 있다(예제 05 참고)."},
 {"id":"QbD","text":"Quality by Design(QbD, ICH Q8)은 목표품질(QTPP)에서 출발해 중요품질특성(CQA)과 "
   "중요공정변수(CPP)를 정의하고 설계공간(Design Space)을 설정한다. AI 최적설계와 잘 맞는다."},
]
print(f"지식베이스 문단 수: {len(KB)}개")


지식베이스 문단 수: 8개


## 3. ⭐ 의미 검색 (RAG 의 R = Retrieval)
질문과 **의미가 비슷한** 문단을 찾습니다. 단어가 정확히 겹치지 않아도
임베딩(벡터) 유사도로 관련 근거를 찾아냅니다. — 여기까지는 **키 없이** 동작.


In [14]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 작고 빠른 임베딩 모델
kb_texts = [d["text"] for d in KB]
kb_vecs = embedder.encode(kb_texts, normalize_embeddings=True)

def retrieve(query, k=3):
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    sims = kb_vecs @ qv                      # 코사인 유사도
    order = np.argsort(-sims)[:k]
    return [(KB[i]["id"], kb_texts[i], float(sims[i])) for i in order]

q = "데이터 무결성 관련해서 FDA 가 자주 지적하는 요건이 뭐야?"
for cid, text, s in retrieve(q):
    print(f"[{cid}] 유사도 {s:.2f}")
    print("  ", textwrap.shorten(text, 90), "\n")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[CSV/CSA] 유사도 0.54
   컴퓨터 시스템 밸리데이션(CSV)은 시스템이 의도대로 동작함을 문서로 입증한다. 최근 FDA 는 위험기반의 CSA(Computer Software [...] 

[ALCOA+] 유사도 0.45
   데이터 무결성의 ALCOA+ 원칙: 데이터는 Attributable(귀속가능), Legible(판독가능), Contemporaneous(동시기록), [...] 

[OOS] 유사도 0.35
   규격이탈(OOS) 결과는 조사(Phase I 실험실 오류 검토 → Phase II 생산 조사)를 거친다. 머신러닝으로 배치 데이터를 학습하면 OOS 를 [...] 



## 4. ⭐ 근거 기반 답변 생성 (RAG 의 G = Generation)
검색된 근거를 프롬프트에 넣어, LLM 이 **출처에 기반해서만** 답하게 합니다.
이렇게 하면 할루시네이션이 줄고, **출처를 밝히는** 규제친화적 답변이 됩니다.


In [15]:
def rag_answer(question, k=3):
    ctx = retrieve(question, k)
    context_block = "\n".join([f"[출처 {cid}] {text}" for cid, text, _ in ctx])
    prompt = f"""너는 제약 GMP/규제 전문가 조수다. 아래 [근거]만 사용해서 한국어로 간결히 답하고,
문장 끝에 사용한 출처 id 를 [출처 ...] 형태로 표시해라. 근거에 없으면 모른다고 말해라.

[근거]
{context_block}

[질문] {question}
[답변]"""
    out = ask_llm(prompt)
    print("근거 문단:", [c[0] for c in ctx])
    print("\n답변:\n", out if out else "(위 프롬프트를 채팅에 붙여넣어 답을 받으세요)")

rag_answer("데이터 무결성 ALCOA+ 요건과 FDA 가 자주 지적하는 부분을 알려줘.")


[키 없음] 아래 프롬프트를 Gemini/Claude 채팅에 붙여넣으세요:

너는 제약 GMP/규제 전문가 조수다. 아래 [근거]만 사용해서 한국어로 간결히 답하고,
문장 끝에 사용한 출처 id 를 [출처 ...] 형태로 표시해라. 근거에 없으면 모른다고 말해라.

[근거]
[출처 ALCOA+] 데이터 무결성의 ALCOA+ 원칙: 데이터는 Attributable(귀속가능), Legible(판독가능), Contemporaneous(동시기록), Original(원본), Accurate(정확) 해야 하며, 확장 요건으로 Complete(완전), Consistent(일관), Enduring(항구), Available(가용) 을 포함한다. FDA 483/경고서한에서 가장 자주 지적되는 영역이다.
[출처 CSV/CSA] 컴퓨터 시스템 밸리데이션(CSV)은 시스템이 의도대로 동작함을 문서로 입증한다. 최근 FDA 는 위험기반의 CSA(Computer Software Assurance) 접근을 권장하여, 문서 부담을 줄이고 중요기능 테스트에 집중하도록 한다.
[출처 생동성] 생물학적 동등성(생동성)은 제네릭이 오리지널과 동등함을 입증한다. AUC 와 Cmax 의 기하평균비 90% 신뢰구간이 80~125% 이내여야 한다.

[질문] 데이터 무결성 ALCOA+ 요건과 FDA 가 자주 지적하는 부분을 알려줘.
[답변]
근거 문단: ['ALCOA+', 'CSV/CSA', '생동성']

답변:
 (위 프롬프트를 채팅에 붙여넣어 답을 받으세요)


## 5. 규제 대응 시나리오 몇 가지 더
같은 지식베이스로 다양한 실무 질문에 근거 기반으로 답할 수 있습니다.
(사내 SOP·가이던스 PDF 를 넣으면 그대로 사내 규제 도우미가 됩니다)


In [16]:
for q in [
    "제네릭 생동성 인정 기준의 신뢰구간 범위는?",
    "안정성 시험으로 유효기간을 어떻게 정하나?",
    "OOS 가 나면 어떤 절차를 밟고 AI 는 어디에 쓰나?",
]:
    print("="*70); print("Q:", q)
    rag_answer(q); print()


Q: 제네릭 생동성 인정 기준의 신뢰구간 범위는?
[키 없음] 아래 프롬프트를 Gemini/Claude 채팅에 붙여넣으세요:

너는 제약 GMP/규제 전문가 조수다. 아래 [근거]만 사용해서 한국어로 간결히 답하고,
문장 끝에 사용한 출처 id 를 [출처 ...] 형태로 표시해라. 근거에 없으면 모른다고 말해라.

[근거]
[출처 OOS] 규격이탈(OOS) 결과는 조사(Phase I 실험실 오류 검토 → Phase II 생산 조사)를 거친다. 머신러닝으로 배치 데이터를 학습하면 OOS 를 조기경보할 수 있다(예제 05 참고).
[출처 공정밸리데이션] 공정 밸리데이션은 3단계로 구성된다: (1)공정설계, (2)공정성능적격성평가(PPQ), (3)지속적 공정검증(CPV). CPV 단계에서 SPC 관리도와 데이터 분석으로 상태를 상시 모니터링한다.
[출처 CSV/CSA] 컴퓨터 시스템 밸리데이션(CSV)은 시스템이 의도대로 동작함을 문서로 입증한다. 최근 FDA 는 위험기반의 CSA(Computer Software Assurance) 접근을 권장하여, 문서 부담을 줄이고 중요기능 테스트에 집중하도록 한다.

[질문] 제네릭 생동성 인정 기준의 신뢰구간 범위는?
[답변]
근거 문단: ['OOS', '공정밸리데이션', 'CSV/CSA']

답변:
 (위 프롬프트를 채팅에 붙여넣어 답을 받으세요)

Q: 안정성 시험으로 유효기간을 어떻게 정하나?
[키 없음] 아래 프롬프트를 Gemini/Claude 채팅에 붙여넣으세요:

너는 제약 GMP/규제 전문가 조수다. 아래 [근거]만 사용해서 한국어로 간결히 답하고,
문장 끝에 사용한 출처 id 를 [출처 ...] 형태로 표시해라. 근거에 없으면 모른다고 말해라.

[근거]
[출처 OOS] 규격이탈(OOS) 결과는 조사(Phase I 실험실 오류 검토 → Phase II 생산 조사)를 거친다. 머신러닝으로 배치 데이터를 학습하면 OOS 를 조기경보할 수 있다(예제 05 참고).
[출처 세척밸리데이션] 세척 밸리데이션은 교차오염을 

## 6. (개념) LLM 에이전트 — 도구를 쓰는 AI
RAG 는 '검색' 이라는 도구 하나를 붙인 것입니다. **에이전트**는 여러 도구
(문서검색, 계산, RDKit 물성계산, 데이터베이스 조회)를 **스스로 골라** 씁니다.
아래는 개념 시연: LLM 이 필요한 도구를 판단하도록 라우팅하는 뼈대입니다.


In [21]:
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

def tool_molecule(smiles):
    """도구: SMILES 물성 계산"""
    m = Chem.MolFromSmiles(smiles)
    if m is None: return "유효하지 않은 SMILES"
    return f"MW={Descriptors.MolWt(m):.1f}, logP={Descriptors.MolLogP(m):.2f}, QED={QED.qed(m):.2f}"

def tool_docs(query):
    """도구: 규제문서 검색"""
    return "; ".join([cid for cid,_,_ in retrieve(query, 2)])

def agent(question):
    # 실제로는 LLM 이 도구선택을 결정. 여기선 키워드로 간단 라우팅(개념 시연)
    if any(ch in question for ch in "()=") or "SMILES" in question:
        smi = question.split()[-1]
        return f"[분자도구 사용] {tool_molecule(smi)}"
    else:
        return f"[문서검색 도구 사용] 관련 근거: {tool_docs(question)}"

print(agent("아스피린 물성 알려줘 CC(=O)Oc1ccccc1C(=O)O"))
print(agent("세척 밸리데이션 요건이 뭐야?"))

[분자도구 사용] MW=180.2, logP=1.31, QED=0.55
[문서검색 도구 사용] 관련 근거: OOS; 세척밸리데이션


## 정리 & 현장 응용
- **대화형 LLM** 은 논문요약·규제해석·코딩까지 연구자의 범용 조수
- **RAG**(검색증강생성): 믿을 문서에서 **먼저 근거를 찾고** LLM 이 그에 기반해 답 → 할루시네이션↓, 출처 명시
- 사내 SOP·FDA 가이던스 PDF 를 지식베이스로 넣으면 **사내 규제 대응 도우미** 완성
- **에이전트**: 문서검색·물성계산 등 여러 도구를 스스로 골라 쓰는 다음 단계
- **면접 한 문장**: "LLM 에 RAG 를 붙여 사내 규제문서에 근거한 답변을 만들면,
  할루시네이션을 줄이고 출처가 명확한 FDA 대응 도우미를 만들 수 있습니다."
